# Scale-aware candidate CNN: two-stage six-hour run
Select **Runtime > Change runtime type > GPU > GPU type: L4**, then Run all. Stage 1 is simulator-supervised pretraining; the six-hour clock applies only to Stage 2 PPO. Drive artifacts are authoritative and rerunning all resumes verified state.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/CNN-RL-improved/scale-aware-cnn-6h-seed0")
PRETRAINING_ROOT = EXPERIMENT_ROOT / "pretraining"
PRETRAINING_DATA_ROOT = PRETRAINING_ROOT / "dataset"
PPO_ROOT = EXPERIMENT_ROOT / "ppo"
for path in (EXPERIMENT_ROOT, PRETRAINING_ROOT, PPO_ROOT):
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
import os, psutil, shutil, subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; select an L4 GPU runtime")
GPU_NAME = torch.cuda.get_device_name(0)
ACCEPTED_GPU_NAMES = {"NVIDIA L4"}
if GPU_NAME not in ACCEPTED_GPU_NAMES:
    raise RuntimeError(f"Expected NVIDIA L4, got {GPU_NAME!r}")
if os.cpu_count() is None or not os.cpu_count() >= 2:
    raise RuntimeError("At least two visible CPU cores are required")
ram = psutil.virtual_memory()
drive_space = shutil.disk_usage("/content/drive/MyDrive")
if ram.total < 10 * 1024**3:
    raise RuntimeError("At least 10 GiB system RAM is required")
if drive_space.free < 10 * 1024**3:
    raise RuntimeError("At least 10 GiB free Drive space is required")
subprocess.run(["nvidia-smi"], check=True)
print({"gpu": GPU_NAME, "torch": torch.__version__, "cuda": torch.version.cuda, "cpu_cores": os.cpu_count(), "ram_gib": ram.total / 1024**3, "drive_free_gib": drive_space.free / 1024**3})


In [ ]:
import subprocess
REPOSITORY = "https://github.com/LMS4681/CNN-RL-Raw-Comparison.git"
RELEASE_TAG = "scale-aware-cnn-6h-v1"
target = Path("/content/CNN-RL-Raw-Comparison").resolve()
if target.exists():
    if target.parent != Path("/content") or not (target / ".git").is_dir():
        raise RuntimeError(f"Unsafe or non-git checkout path: {target}")
else:
    subprocess.run(["git", "clone", "--branch", RELEASE_TAG, "--depth", "1", REPOSITORY, str(target)], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=target, text=True).strip()
tag_head = subprocess.check_output(["git", "rev-list", "-n", "1", RELEASE_TAG], cwd=target, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=target, text=True)
if head != tag_head or dirty:
    raise RuntimeError(f"Checkout is not the clean pinned tag: HEAD={head}, tag={tag_head}, dirty={dirty!r}")
print("Pinned checkout:", head)


In [ ]:
import importlib.metadata, json, subprocess, sys
def child_torch_snapshot():
    source = 'import json, torch; print(json.dumps({"version": torch.__version__, "file": torch.__file__, "cuda": torch.version.cuda, "available": torch.cuda.is_available()}))'
    return json.loads(subprocess.check_output([sys.executable, "-c", source], text=True))
def pip_check_conflicts():
    result = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
    return {line.strip() for line in (result.stdout + result.stderr).splitlines() if line.strip()}
before_torch = child_torch_snapshot()
before_pip_conflicts = pip_check_conflicts()
lock_path = target / "AllocRL" / "requirements-comparison.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--require-hashes", "-r", str(lock_path)], check=True)
after_torch = child_torch_snapshot()
after_pip_conflicts = pip_check_conflicts()
new_pip_conflicts = after_pip_conflicts - before_pip_conflicts
if after_torch != before_torch:
    raise RuntimeError(f"Colab Torch stack changed: {before_torch} -> {after_torch}")
if new_pip_conflicts:
    raise RuntimeError("Dependency install introduced conflicts: " + "; ".join(sorted(new_pip_conflicts)))


In [ ]:
import hashlib, json, os
ALLOC_RL = target / "AllocRL"
os.chdir(ALLOC_RL)
expected_hashes = {
    "data/fixed_eval_scenarios.json": "913cac9046dec8164ef65da60275522f7127de5ea775b1c5a6b6aac255716271",
    "data/data_split_manifest.json": "601bd6143ed8890577e5ff34921241d36fd6a0e99c4bdab4e26152ab168178f8",
    "requirements-comparison.txt": "37634576e34043d169cf24bfc0cc2261818dc65b9358d4b9b2e46ab614d0bdda",
    "configs/candidate_pretrain_seed0.json": "a15b8dc3510bb0fbeab216d44f67d075fbd9fec05ebdd4ba24bf2aaed90fbd70",
    "configs/improved_cnn_6h_seed0.json": "87da6e18f7892b2a22e57e0244f69e8e87bd0b5f38bc9aef8e738b65e50d1193",
}
for relative, expected in expected_hashes.items():
    actual = hashlib.sha256((ALLOC_RL / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Immutable input hash mismatch for {relative}: {actual} != {expected}")
PPO_CONFIG_SHA256 = expected_hashes["configs/improved_cnn_6h_seed0.json"]
print("Immutable hashes verified")


In [ ]:
import json, shutil, subprocess, sys
from pretraining.dataset import load_pretraining_shard, read_dataset_manifest
LOCAL_DATASET = Path("/content/pretraining-dataset")
if PRETRAINING_DATA_ROOT.exists() and not LOCAL_DATASET.exists():
    shutil.copytree(PRETRAINING_DATA_ROOT, LOCAL_DATASET)
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)
dataset_command = [sys.executable, "-m", "pretraining.dataset", "--config", str(ALLOC_RL / "configs/candidate_pretrain_seed0.json"), "--output-dir", str(LOCAL_DATASET), "--progress-mirror-dir", str(PRETRAINING_DATA_ROOT)]
subprocess.run(dataset_command, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
manifest_path = LOCAL_DATASET / "dataset_manifest.json"
manifest = read_dataset_manifest(manifest_path)
for split in manifest["splits"].values():
    for entry in split["shards"]:
        load_pretraining_shard(LOCAL_DATASET, manifest, entry)
PRETRAINING_DATA_ROOT.mkdir(parents=True, exist_ok=True)
for item in LOCAL_DATASET.rglob("*"):
    if item.is_file() and item.name != "dataset_manifest.json":
        destination = PRETRAINING_DATA_ROOT / item.relative_to(LOCAL_DATASET)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, destination)
shutil.copy2(manifest_path, PRETRAINING_DATA_ROOT / "dataset_manifest.json")
print("Verified dataset states:", {name: split["state_count"] for name, split in manifest["splits"].items()})


In [ ]:
import subprocess, sys
from pretraining.transfer import verify_pretraining_artifacts
PRETRAIN_CHECKPOINT = PRETRAINING_ROOT / "candidate_encoder_pretrained.pt"
PRETRAIN_COMPLETE = PRETRAINING_ROOT / "PRETRAINING_COMPLETE.json"
pretraining_command = [sys.executable, "-m", "pretraining.train_encoder", "--config", str(ALLOC_RL / "configs/candidate_pretrain_seed0.json"), "--dataset-root", str(LOCAL_DATASET), "--output-dir", str(PRETRAINING_ROOT), "--device", "cuda"]
if PRETRAIN_COMPLETE.is_file():
    verified_stage1 = verify_pretraining_artifacts(PRETRAIN_CHECKPOINT, PRETRAIN_COMPLETE)
else:
    subprocess.run(pretraining_command, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    verified_stage1 = verify_pretraining_artifacts(PRETRAIN_CHECKPOINT, PRETRAIN_COMPLETE)
expected_pretrain_config_sha = expected_hashes["configs/candidate_pretrain_seed0.json"]
expected_manifest_sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
if verified_stage1.receipt.config_sha256 != expected_pretrain_config_sha or verified_stage1.receipt.manifest_sha256 != expected_manifest_sha:
    raise RuntimeError("Stage 1 receipt does not match the pinned config and dataset manifest")
print("Stage 1 verified:", verified_stage1.receipt)


In [ ]:
import subprocess, sys, tempfile
with tempfile.TemporaryDirectory(prefix="scale-aware-smoke-") as smoke_dir:
    smoke_command = [sys.executable, "-m", "pretraining.two_stage_smoke", "--checkpoint", str(PRETRAIN_CHECKPOINT), "--complete", str(PRETRAIN_COMPLETE), "--output-dir", smoke_dir, "--timesteps", "32", "--device", "cuda"]
    subprocess.run(smoke_command, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    smoke_result = json.loads((Path(smoke_dir) / "SMOKE_COMPLETE.json").read_text(encoding="utf-8"))
    if not all(smoke_result[key] for key in ("extractor_unchanged_while_frozen", "extractor_changed_after_unfreeze", "nonzero_extractor_gradient")):
        raise RuntimeError(f"Two-stage smoke failed: {smoke_result}")
    if abs(smoke_result["extractor_lr_ratio"] - 0.1) > 1e-12:
        raise RuntimeError(f"Smoke LR ratio failed: {smoke_result}")
print("32-step Stage 2 smoke verified")


In [ ]:
import json, os, subprocess, sys
ppo_command = [sys.executable, "-u", "train.py", "--data-dir", "./data", "--output-dir", str(PPO_ROOT), "--extractor", "candidate-cnn", "--state-context", "full", "--seed", "0", "--timesteps", "2000000000", "--max-training-seconds", "21600", "--lr", "0.0001", "--lr-schedule", "linear", "--lr-final", "0.00001", "--lr-decay-steps", "1000000", "--pretrained-extractor", str(PRETRAIN_CHECKPOINT), "--pretraining-complete", str(PRETRAIN_COMPLETE), "--require-pretrained-extractor", "--freeze-extractor-steps", "50000", "--extractor-lr-scale", "0.1", "--n-envs", "8", "--vec-env", "subproc", "--n-steps", "120", "--batch-size", "64", "--n-epochs", "5", "--gamma", "1.0", "--gae-lambda", "0.98", "--checkpoint-freq", "10000", "--wall-clock-heartbeat-seconds", "300", "--holdout-eval-freq", "50000", "--holdout-selection-count", "5", "--monthly-jitter", "20", "--empirical-profile-probability", "0.2", "--device", "cuda", "--eval-scenarios", "./data/fixed_eval_scenarios.json", "--final-holdout-report", "--comparison-config-sha256", PPO_CONFIG_SHA256, "--auto-resume", "--no-export-onnx"]
state_path = PPO_ROOT / "run_state.json"
command_to_run = list(ppo_command)
if state_path.is_file():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    if state.get("status") == "complete":
        exact_checkpoint = PPO_ROOT / "checkpoints" / state["last_checkpoint_file"]
        command_to_run = [item for item in ppo_command if item != "--auto-resume"] + ["--resume-from", str(exact_checkpoint), "--finalize-complete-state"]
subprocess.run(command_to_run, check=True, cwd=ALLOC_RL, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
from IPython.display import Markdown, display
artifact_paths = [PRETRAIN_COMPLETE, PRETRAINING_ROOT / "pretraining_metrics.json", PPO_ROOT / "run_state.json", PPO_ROOT / "progress_timing.csv", PPO_ROOT / "training_completion.json", PPO_ROOT / "evaluation_csv.csv", PPO_ROOT / "evaluation_scenarios.csv"]
for artifact in artifact_paths:
    if artifact.is_file():
        print(f"\n===== {artifact} =====")
        print(artifact.read_text(encoding="utf-8")[:20000])
    else:
        print(f"MISSING: {artifact}")
state = json.loads((PPO_ROOT / "run_state.json").read_text(encoding="utf-8")) if (PPO_ROOT / "run_state.json").is_file() else None
exact_resume = None if state is None else PPO_ROOT / "checkpoints" / state["last_checkpoint_file"]
print("\nStage 1 resume command:", " ".join(pretraining_command))
print("Stage 2 auto-resume command:", " ".join(ppo_command))
if exact_resume is not None:
    print("Stage 2 exact resume command checkpoint:", exact_resume)
display(Markdown(f"**Experiment root:** `{EXPERIMENT_ROOT}`"))
